In [9]:
import os, sys
project_root = '/root/work/tvm-ansor'
os.environ['TVM_HOME'] = project_root
os.environ['TVM_LIBRARY_PATH'] = f'{project_root}/build-release'
if f'{project_root}/python' not in sys.path:
    sys.path.insert(0, f'{project_root}/python')
sys.path = [p for p in sys.path if not p.startswith(f'{project_root}/build')]
sys.path.append(f'{project_root}/build-release')
os.environ['LD_LIBRARY_PATH'] = f'{project_root}/build-release:' + os.environ.get('LD_LIBRARY_PATH', '')

import numpy as np, math, json, copy, random, tempfile
from collections import defaultdict
from types import SimpleNamespace
import tvm
from tvm import relay, auto_scheduler
from tvm.auto_scheduler.feature import get_per_store_features_from_states
from tvm.auto_scheduler.measure import recover_measure_input
import util_manager
from util_manager import PathManager, get_network

TARGET = tvm.target.Target('cuda')
HW = {
    'warp_size': 32,
    'max_vthread_extent': 8,
    'max_threads_per_block': 1024,
    'max_shared_memory_per_block': 49152,
    'max_innermost_split_factor': 64,
}
AUTO_UNROLL_CONFIGS = [0, 16, 64, 512, 1024]
print('Setup done.')

Setup done.


In [10]:
args = SimpleNamespace(network='resnet_18', batch_size=1, dtype='float32', layout='NHWC', timenow=None, json=None)
mod, params, input_shape, output_shape = get_network(args.network, args.batch_size, args.layout, dtype=args.dtype)
path_manager = PathManager(args.network, input_shape, args, None, json='/root/work/tvm-ansor/gallery/logs_json/tmp.json')
tasks, task_weights = path_manager.tasks_pkl_use()
tasks, task_weights = zip(*sorted(zip(tasks, task_weights), key=lambda x: x[0].desc))
tasks, task_weights = list(tasks), list(task_weights)
print(f'Tasks: {len(tasks)}')

LOG_FILE = '/root/work/tvm-ansor/gallery/logs_json/resnet_18/resnet_18-B1.json'
records = []
with open(LOG_FILE) as f:
    for line in f:
        line = line.strip()
        if line: records.append(json.loads(line))
groups = defaultdict(list)
for rec in records:
    groups[rec['i'][0][0]].append(rec)
groups = dict(groups)
print(f'Records: {len(records)}, Unique tasks: {len(groups)}')

Getting network resnet_18...

Using json : /root/work/tvm-ansor/gallery/logs_json/tmp.json
Load tasks from /root/work/tvm-ansor/gallery/ansor_tasks_pkl/resnet_18-(1,224,224,3).pkl
Tasks: 24
Records: 2408, Unique tasks: 24


In [11]:
def get_divisors(n):
    if n <= 0: return [1]
    divs = set()
    for i in range(1, int(math.isqrt(n)) + 1):
        if n % i == 0:
            divs.add(i); divs.add(n // i)
    return sorted(divs)

def record_to_task_and_state(record):
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        f.write(json.dumps(record) + '\n'); tmp = f.name
    try:
        inputs, _ = zip(*auto_scheduler.load_records(tmp))
        inp = recover_measure_input(inputs[0], rebuild_state=True)
        return inp.task, inp.state
    finally:
        os.unlink(tmp)

def validate_via_feature(record):
    try:
        task, state = record_to_task_and_state(record)
    except Exception as e:
        return False, f'recovery: {e}'
    try:
        fea = get_per_store_features_from_states([state], task)[0]
        if fea.shape[0] == 1 and np.all(fea == 0):
            return False, 'all-zero'
        return True, f'OK {fea.shape}'
    except Exception as e:
        return False, f'exc: {e}'

def inject_params(record, assignments):
    rec = copy.deepcopy(record)
    steps = rec['i'][1][1]
    for step_idx, val in assignments:
        s = steps[step_idx]
        if s[0] == 'SP': s[4] = list(val)
        elif s[0] == 'PR': s[3] = f'auto_unroll_max_step${val}'
    return rec

print('Utils loaded.')

Utils loaded.


In [12]:
import re
from tvm import tir

_s2m = tvm.get_global_func("driver.schedule_to_module")
_verify_gpu = tvm.get_global_func("tir.analysis.verify_gpu_code")

GPU_PASSES = tvm.transform.Sequential([
    tir.transform.InjectPrefetch(),
    tir.transform.StorageFlatten(64, False),
    tir.transform.NarrowDataType(32),
    tir.transform.Simplify(),
    tir.transform.VectorizeLoop(True),
    tir.transform.InjectVirtualThread(),
    tir.transform.StorageRewrite(),
    tir.transform.Simplify(),
])

GPU_HW = {
    'max_shared_memory_per_block': 49152,
    'max_local_memory_per_block': 2**31 - 1,
    'max_threads_per_block': 1024,
    'max_vthread': 8,
    'max_vector_bytes': 16,
    'warp_size': 32,
    'max_innermost_split_factor': 64,
}
AUTO_UNROLL_CONFIGS = [0, 16, 64, 512, 1024]

def lower_with_gpu_passes(task, state):
    """feature.cc와 동일: ScheduleToModule + GPU pass pipeline."""
    sch, tensors = task.compute_dag.apply_steps_from_state(state)
    mod = _s2m(sch, tensors, "main", {})
    return GPU_PASSES(mod)

def parse_tir_constraints(tir_str):
    """TIR 문자열에서 per-kernel GPU 제약값 추출 (VerifyGPUCode 동일)."""
    kernels = []
    cur = None
    max_vthread_s = 1

    for line in tir_str.split('\n'):
        if re.search(r'launch_thread\("blockIdx\.\w+",\s*\d+\)', line):
            if cur is not None:
                kernels.append(cur)
            cur = {'thread_per_block': 1, 'threads': {}, 'shared_bytes': 0,
                   'local_bytes': 0, 'vthread': 1}

        if cur is None:
            continue

        mt = re.search(r'launch_thread\("threadIdx\.(\w+)",\s*(\d+)\)', line)
        if mt:
            dim, ext = mt.group(1), int(mt.group(2))
            if dim not in cur['threads']:
                cur['threads'][dim] = ext
                cur['thread_per_block'] *= ext

        mv = re.search(r'launch_thread\("vthread\w*",\s*(\d+)\)', line)
        if mv:
            v = int(mv.group(1))
            cur['vthread'] *= v
            cur['thread_per_block'] *= v

        ms = re.search(r'allocate\(\[(\d+)\],\s*"([^"]+)",\s*"shared"', line)
        if ms:
            cur['shared_bytes'] += int(ms.group(1)) * (tvm.DataType(ms.group(2)).bits // 8)

        ml = re.search(r'allocate\(\[(\d+)\],\s*"([^"]+)",\s*"local"', line)
        if ml:
            cur['local_bytes'] += int(ml.group(1)) * (tvm.DataType(ml.group(2)).bits // 8)

        mvs = re.search(r'for\s+([\w,\s]+)\s+in\s+T\.grid\(([^)]+)\)', line)
        if mvs:
            vars_ = [v.strip() for v in mvs.group(1).split(',')]
            if 'vthread_s' in vars_:
                exts = [int(x.strip()) for x in mvs.group(2).split(',')]
                max_vthread_s = max(max_vthread_s, exts[vars_.index('vthread_s')])
        mvs2 = re.search(r'for\s+vthread_s\s+in\s+(?:range|T\.serial)\((\d+)\)', line)
        if mvs2:
            max_vthread_s = max(max_vthread_s, int(mvs2.group(1)))

    if cur is not None:
        kernels.append(cur)
    return kernels, max_vthread_s

def extract_gpu_constraints(task, state):
    """State → GPU pass pipeline → (kernels, max_vthread_s)."""
    mod = lower_with_gpu_passes(task, state)
    return parse_tir_constraints(str(mod))

def check_hw_limits(kernels, max_vts, hw=GPU_HW):
    """Per-kernel 제약 체크. 위반 시 (False, reason) 반환."""
    if max_vts > hw['max_vthread']:
        return False, f"vthread.s={max_vts}"
    for i, k in enumerate(kernels):
        if k['thread_per_block'] > hw['max_threads_per_block']:
            return False, f"K{i}:thread={k['thread_per_block']}"
        if k['shared_bytes'] > hw['max_shared_memory_per_block']:
            return False, f"K{i}:shared={k['shared_bytes']}"
        if k['local_bytes'] > hw['max_local_memory_per_block']:
            return False, f"K{i}:local={k['local_bytes']}"
        if k['vthread'] > hw['max_vthread']:
            return False, f"K{i}:vthread={k['vthread']}"
    return True, 'pass'

def verify_state_exact(task, state, hw=GPU_HW):
    """ScheduleToModule + GPU passes + VerifyGPUCode 정확 체크."""
    try:
        mod = lower_with_gpu_passes(task, state)
        for gv, fn in mod.functions.items():
            if isinstance(fn, tvm.tir.PrimFunc):
                ok = _verify_gpu(fn, {
                    "max_shared_memory_per_block": hw['max_shared_memory_per_block'],
                    "max_local_memory_per_block": hw['max_local_memory_per_block'],
                    "max_threads_per_block": hw['max_threads_per_block'],
                    "max_vector_bytes": hw['max_vector_bytes'],
                    "max_vthread": hw['max_vthread'],
                })
                if not ok:
                    return False
        return True
    except Exception:
        return False

print('Phase 1: TIR constraint extraction loaded.')성.

    build() 시 N+1회 InferBound probe로 shared memory 공식 도출.
    generate_random_params()는 TVM 호출 없이 valid 파라미터 생성.

    제약 (spatial 4-way split: [l0, l1, l2, l3]):
      - l0 (innermost) ≤ 64
      - product(l3_i) ≤ 8  (vthread)
      - 32 ≤ product(l2_i) ≤ 1024  (thread, 조건부)
      - shared memory ≤ 49152 bytes
      - factor: 각 length는 remaining extent의 약수

    생성 순서: l3 → l2 → l0 → l1 (제약 우선)
    """

    def __init__(self, record, hw=None):
        self.record = record
        self.hw = hw or HW
        self.built = False
        self._parse()

    def _parse(self):
        steps = self.record['i'][1][1]
        stage_sps = defaultdict(list)
        for i, s in enumerate(steps):
            if s[0] == 'SP':
                stage_sps[s[1]].append({
                    'step_idx': i, 'stage_id': s[1], 'iter_id': s[2],
                    'extent': s[3], 'lengths': s[4], 'inner_to_outer': s[5],
                })

        self.main_stage_id = None
        max4 = 0
        for sid, sps in stage_sps.items():
            n4 = sum(1 for sp in sps if len(sp['lengths']) == 4)
            if n4 > max4:
                max4 = n4
                self.main_stage_id = sid

        self.spatial_sps = []
        self.reduce_sps = []
        self.thread_bind_sps = []
        self.other_sps = []
        self.pr_steps = []

        if self.main_stage_id is not None:
            for sp in stage_sps[self.main_stage_id]:
                if len(sp['lengths']) == 4:
                    self.spatial_sps.append(sp)
                elif len(sp['lengths']) == 2:
                    self.reduce_sps.append(sp)
                else:
                    self.other_sps.append(sp)
            for sid, sps in stage_sps.items():
                if sid != self.main_stage_id:
                    for sp in sps:
                        if len(sp['lengths']) == 1:
                            self.thread_bind_sps.append(sp)
                        else:
                            self.other_sps.append(sp)

        for i, s in enumerate(steps):
            if s[0] == 'PR' and 'auto_unroll_max_step' in s[3]:
                self.pr_steps.append({
                    'step_idx': i, 'stage_id': s[1],
                    'value': int(s[3].split('$')[1]),
                })

        spatial_exts = [sp['extent'] for sp in self.spatial_sps]
        self.total_space = math.prod(spatial_exts) if spatial_exts else 0
        self.check_min_thread = self.total_space > self.hw['warp_size'] * 2

    # ── Probe 기반 Shared Memory 공식 구축 ──

    def _get_shared_sizes(self, s_lengths, r_lengths):
        rec = copy.deepcopy(self.record)
        steps = rec['i'][1][1]
        for sp, ln in zip(self.spatial_sps, s_lengths):
            steps[sp['step_idx']][4] = list(ln)
        for sp, ln in zip(self.reduce_sps, r_lengths):
            steps[sp['step_idx']][4] = list(ln)
        with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
            f.write(json.dumps(rec) + '\n')
            tmp = f.name
        try:
            inputs, _ = zip(*auto_scheduler.load_records(tmp))
            inp = recover_measure_input(inputs[0], rebuild_state=True)
            bs = inp.task.compute_dag.infer_bound_from_state(inp.state)
            sizes = {}
            for si in range(len(bs.stages)):
                name = str(bs.stages[si].op.name)
                if '.shared' not in name:
                    continue
                total = 1
                for it in bs.stages[si].iters:
                    if it.range is not None:
                        total *= int(it.range.extent)
                sizes[name] = total
            return sizes
        finally:
            os.unlink(tmp)

    def build(self):
        if not self.spatial_sps:
            self.shared_formulas = []
            self.built = True
            return

        n_s = len(self.spatial_sps)
        n_r = len(self.reduce_sps)

        base_s = [[1] * 4 for _ in range(n_s)]
        base_r = [[1] * len(sp['lengths']) for sp in self.reduce_sps]
        base = self._get_shared_sizes(base_s, base_r)

        probe_results = {}
        for i in range(n_s):
            if self.spatial_sps[i]['extent'] < 2:
                continue
            ps = [list(x) for x in base_s]
            ps[i][0] = 2
            probed = self._get_shared_sizes(ps, base_r)
            for buf in base:
                if base[buf] > 0 and probed.get(buf, base[buf]) != base[buf]:
                    probe_results.setdefault(('spatial', i), {})[buf] = probed[buf] / base[buf]

        for i in range(n_r):
            if self.reduce_sps[i]['extent'] < 2:
                continue
            pr = [list(x) for x in base_r]
            pr[i][0] = 2
            probed = self._get_shared_sizes(base_s, pr)
            for buf in base:
                if base[buf] > 0 and probed.get(buf, base[buf]) != base[buf]:
                    probe_results.setdefault(('reduce', i), {})[buf] = probed[buf] / base[buf]

        self.shared_formulas = []
        for buf_name in base:
            contribs = []
            for (kind, idx), ratios in probe_results.items():
                if buf_name in ratios:
                    contribs.append((kind, idx, ratios[buf_name]))
            self.shared_formulas.append({
                'name': buf_name,
                'base': base[buf_name],
                'contributors': contribs,
                'dtype_bytes': 4,
            })

        self.built = True

    def estimate_shared_memory(self, spatial_tiles, reduce_tiles):
        total = 0
        for f in self.shared_formulas:
            val = f['base']
            for kind, idx, ratio in f['contributors']:
                t = spatial_tiles[idx] if kind == 'spatial' else reduce_tiles[idx]
                if abs(ratio - 2.0) < 0.01:
                    val *= t
                else:
                    stride = max(1, round(ratio - 1))
                    val *= max(1, stride * t - stride + 1)
            total += val * f['dtype_bytes']
        return total

    # ── 랜덤 파라미터 생성 ──
    # 순서: l3(vthread) → l2(thread) → l0(innermost) → l1(inner)

    def generate_random_params(self, rng=None, max_retries=3000):
        if rng is None:
            rng = random.Random()
        if not self.spatial_sps:
            return self._gen_simple(rng)
        for _ in range(max_retries):
            r = self._try_gen(rng)
            if r is not None:
                return r
        raise RuntimeError('generation failed')

    def _try_gen(self, rng):
        hw = self.hw
        n = len(self.spatial_sps)
        extents = [sp['extent'] for sp in self.spatial_sps]

        # Phase 1: l3 (vthread) — product ≤ 8
        l3s = []
        vt_budget = hw['max_vthread_extent']
        for i in range(n):
            divs = [d for d in get_divisors(extents[i]) if d <= vt_budget]
            if not divs:
                return None
            v = rng.choice(divs)
            l3s.append(v)
            vt_budget //= v

        # Phase 2: l2 (thread) — product ∈ [warp_size, max_threads]
        rem_after_l3 = [extents[i] // l3s[i] for i in range(n)]
        l2s = [0] * n
        th_product = 1
        th_min = hw['warp_size'] if self.check_min_thread else 1
        th_max = hw['max_threads_per_block']

        order = list(range(n))
        rng.shuffle(order)
        for rank, i in enumerate(order):
            divs = [d for d in get_divisors(rem_after_l3[i]) if th_product * d <= th_max]
            if not divs:
                return None
            sps_left = n - rank - 1
            needed = math.ceil(th_min / th_product)
            if sps_left > 0:
                needed_here = max(1, math.ceil(needed ** (1.0 / (sps_left + 1))))
            else:
                needed_here = needed
            preferred = [d for d in divs if d >= needed_here]
            l2s[i] = rng.choice(preferred) if preferred else rng.choice(divs)
            th_product *= l2s[i]

        if th_product < th_min or th_product > th_max:
            return None

        # Phase 3: l0 (innermost) — each ≤ 64
        mi = hw['max_innermost_split_factor']
        rem_after_l23 = [rem_after_l3[i] // l2s[i] for i in range(n)]
        l0s = []
        for i in range(n):
            divs = [d for d in get_divisors(rem_after_l23[i]) if d <= mi]
            if not divs:
                return None
            l0s.append(rng.choice(divs))

        # Phase 4: l1 (inner) — unconstrained
        rem_after_l023 = [rem_after_l23[i] // l0s[i] for i in range(n)]
        l1s = []
        for i in range(n):
            divs = get_divisors(rem_after_l023[i])
            l1s.append(rng.choice(divs))

        spatial_lengths = [[l0s[i], l1s[i], l2s[i], l3s[i]] for i in range(n)]

        # Reduce splits
        reduce_lengths = []
        for sp in self.reduce_sps:
            ln = self._fill_reduce(sp['extent'], len(sp['lengths']), mi, rng)
            if ln is None:
                return None
            reduce_lengths.append(ln)

        # Shared memory check
        if self.shared_formulas:
            s_tiles = [math.prod(sl) for sl in spatial_lengths]
            r_tiles = [rl[0] for rl in reduce_lengths]
            est = self.estimate_shared_memory(s_tiles, r_tiles)
            if est > hw['max_shared_memory_per_block']:
                return None

        # Assemble
        assignments = []
        for sp, ln in zip(self.spatial_sps, spatial_lengths):
            assignments.append((sp['step_idx'], ln))
        for sp, ln in zip(self.reduce_sps, reduce_lengths):
            assignments.append((sp['step_idx'], ln))
        for sp in self.thread_bind_sps:
            assignments.append((sp['step_idx'], sp['lengths']))
        for sp in self.other_sps:
            assignments.append((sp['step_idx'], sp['lengths']))
        for pr in self.pr_steps:
            assignments.append((pr['step_idx'], rng.choice(AUTO_UNROLL_CONFIGS)))
        return assignments

    def _fill_reduce(self, extent, n_slots, max_innermost, rng):
        lengths = []
        remaining = extent
        for slot in range(n_slots):
            divs = get_divisors(remaining)
            if slot == 0:
                divs = [d for d in divs if d <= max_innermost]
            if not divs:
                return None
            v = rng.choice(divs)
            lengths.append(v)
            remaining //= v
        return lengths

    def _gen_simple(self, rng):
        a = []
        for sp in self.reduce_sps + self.thread_bind_sps + self.other_sps:
            a.append((sp['step_idx'], sp['lengths']))
        for pr in self.pr_steps:
            a.append((pr['step_idx'], rng.choice(AUTO_UNROLL_CONFIGS)))
        return a

    def summary(self):
        lines = [f'Main stage: {self.main_stage_id}']
        lines.append(f'Spatial SPs: {[(sp["extent"], sp["lengths"]) for sp in self.spatial_sps]}')
        lines.append(f'Reduce SPs: {[(sp["extent"], sp["lengths"]) for sp in self.reduce_sps]}')
        lines.append(f'Thread bind: {len(self.thread_bind_sps)}, PR: {len(self.pr_steps)}')
        lines.append(f'Total space: {self.total_space}, check_min_thread: {self.check_min_thread}')
        if self.built:
            lines.append(f'Shared formulas ({len(self.shared_formulas)} bufs):')
            for f in self.shared_formulas:
                contribs = [(f'{k}[{i}] r={r:.2f}') for k, i, r in f['contributors']]
                lines.append(f'  {f["name"]}: base={f["base"]}, {contribs}')
        return '\n'.join(lines)

print('ConstraintSystem defined.')

ConstraintSystem defined.


In [13]:
class ConstraintSystem:
    """Probe-based GPU constraint formulas for a single task/schedule.

    Usage:
        cs = ConstraintSystem(record)
        ok = cs.build()          # N+1 lowerings → formula derivation
        rec = cs.generate(rng)   # random valid params
    """

    def __init__(self, record, hw=GPU_HW):
        self.record = record
        self.steps = record['i'][1][1]
        self.hw = hw
        self._parse_steps()

    # ── schedule parsing ──

    def _parse_steps(self):
        self.sps = []
        self.prs = []
        for i, s in enumerate(self.steps):
            if s[0] == 'SP':
                self.sps.append({
                    'idx': i, 'stage': s[1], 'iter': s[2],
                    'extent': s[3], 'nlens': len(s[4]), 'orig': list(s[4]),
                })
            elif s[0] == 'PR' and 'auto_unroll_max_step' in str(s[3]):
                self.prs.append({'idx': i, 'stage': s[1]})

        # dimensions: (sp_index, length_position)
        self.dims = []
        for si, sp in enumerate(self.sps):
            for li in range(sp['nlens']):
                self.dims.append((si, li))

    def _smallest_factor(self, n, gt=1):
        for p in range(gt + 1, n + 1):
            if n % p == 0:
                return p
        return n if n > gt else gt + 1

    # ── probing ──

    def _make_record(self, dim_vals):
        """dim_vals: {(si, li): value}. Unspecified dims default to 1."""
        rec = copy.deepcopy(self.record)
        for sp in self.sps:
            si = self.sps.index(sp)
            rec['i'][1][1][sp['idx']][4] = [dim_vals.get((si, li), 1) for li in range(sp['nlens'])]
        return rec

    def _extract(self, rec):
        task, state = record_to_task_and_state(rec)
        kernels, vts = extract_gpu_constraints(task, state)
        return {'kernels': kernels, 'vts': vts, 'task': task, 'state': state}

    def build(self):
        """N+1 probing → formulas."""
        # base: all lengths = 1
        self.base = self._extract(self._make_record({}))
        self.n_kernels = len(self.base['kernels'])

        # probe each dimension
        self.probes = {}
        for si, sp in enumerate(self.sps):
            ext = sp['extent']
            for li in range(sp['nlens']):
                pv = self._smallest_factor(ext)
                if pv > ext:
                    pv = ext
                self.probes[(si, li)] = {
                    'pv': pv,
                    'result': self._extract(self._make_record({(si, li): pv})),
                }

        # derive formulas: per-kernel, per-metric
        metrics = ['shared_bytes', 'local_bytes', 'thread_per_block']
        self.k_formulas = []  # per-kernel list of {metric: {base, contribs}}
        for ki in range(self.n_kernels):
            kf = {}
            for m in metrics:
                base_val = self.base['kernels'][ki][m]
                contribs = []
                for dim, pd in self.probes.items():
                    if ki >= len(pd['result']['kernels']):
                        continue
                    pval = pd['result']['kernels'][ki][m]
                    if base_val > 0 and pval != base_val:
                        contribs.append({'dim': dim, 'pv': pd['pv'],
                                         'ratio': pval / base_val})
                    elif base_val == 0 and pval > 0:
                        contribs.append({'dim': dim, 'pv': pd['pv'], 'ratio': float('inf')})
                kf[m] = {'base': base_val, 'contribs': contribs}
            self.k_formulas.append(kf)

        # vthread.s formula
        base_vts = self.base['vts']
        vts_contribs = []
        for dim, pd in self.probes.items():
            pvts = pd['result']['vts']
            if base_vts > 0 and pvts != base_vts:
                vts_contribs.append({'dim': dim, 'pv': pd['pv'], 'ratio': pvts / base_vts})
            elif base_vts == 0 and pvts > 0:
                vts_contribs.append({'dim': dim, 'pv': pd['pv'], 'ratio': float('inf')})
        self.vts_formula = {'base': base_vts, 'contribs': vts_contribs}

        return True

    # ── formula evaluation ──

    def _eval_formula(self, formula, dim_vals):
        v = formula['base']
        if v == 0:
            for c in formula['contribs']:
                if dim_vals.get(c['dim'], 1) > 1:
                    return float('inf')
            return 0
        for c in formula['contribs']:
            f = dim_vals.get(c['dim'], 1)
            if c['ratio'] == float('inf'):
                if f > 1:
                    return float('inf')
            else:
                pv = c['pv']
                g = 1 + (c['ratio'] - 1) * (f - 1) / max(pv - 1, 1)
                v *= g
        return v

    def estimate(self, dim_vals):
        """Return dict of estimated constraint values."""
        est = {'vts': self._eval_formula(self.vts_formula, dim_vals)}
        for ki, kf in enumerate(self.k_formulas):
            for m in ['shared_bytes', 'local_bytes', 'thread_per_block']:
                est[f'K{ki}_{m}'] = self._eval_formula(kf[m], dim_vals)
        return est

    def formula_check(self, dim_vals):
        """Fast constraint check using formulas."""
        vts = self._eval_formula(self.vts_formula, dim_vals)
        if vts > self.hw['max_vthread']:
            return False, f'vts={vts:.0f}'
        for ki, kf in enumerate(self.k_formulas):
            th = self._eval_formula(kf['thread_per_block'], dim_vals)
            if th > self.hw['max_threads_per_block']:
                return False, f'K{ki}:th={th:.0f}'
            sh = self._eval_formula(kf['shared_bytes'], dim_vals)
            if sh > self.hw['max_shared_memory_per_block']:
                return False, f'K{ki}:sh={sh:.0f}'
            lo = self._eval_formula(kf['local_bytes'], dim_vals)
            if lo > self.hw['max_local_memory_per_block']:
                return False, f'K{ki}:lo={lo:.0f}'
        return True, 'pass'

    # ── random generation ──

    def generate(self, rng=None, max_attempts=200):
        """Generate a random valid record."""
        if rng is None:
            rng = random.Random()
        for _ in range(max_attempts):
            dv = self._fill_greedy(rng)
            if dv is None:
                continue
            ok, _ = self.formula_check(dv)
            if ok:
                return self._to_record(dv, rng)
        return None

    def _fill_greedy(self, rng):
        """Greedy filling: constrained dimensions first.
        Order: l3(vthread) → thread-bind SPs → l2(thread) → l0(innermost) → l1(rest)"""
        dv = {}
        max_isf = self.hw['max_innermost_split_factor']

        for si, sp in enumerate(self.sps):
            remaining = sp['extent']
            for li in range(sp['nlens']):
                divs = sorted(get_divisors(remaining))
                if li == 0:
                    divs = [d for d in divs if d <= max_isf] or [1]
                val = rng.choice(divs)
                dv[(si, li)] = val
                remaining //= val

        ok, _ = self.formula_check(dv)
        if not ok:
            return None
        return dv

    def _to_record(self, dim_vals, rng):
        assignments = []
        for si, sp in enumerate(self.sps):
            lengths = tuple(dim_vals[(si, li)] for li in range(sp['nlens']))
            assignments.append((sp['idx'], lengths))
        for pr in self.prs:
            assignments.append((pr['idx'], rng.choice(AUTO_UNROLL_CONFIGS)))
        return inject_params(self.record, assignments)

    # ── diagnostics ──

    def summary(self):
        lines = [f'SPs: {len(self.sps)}, PRs: {len(self.prs)}, Dims: {len(self.dims)}, Kernels: {self.n_kernels}']
        for ki, kf in enumerate(self.k_formulas):
            base_th = kf['thread_per_block']['base']
            base_sh = kf['shared_bytes']['base']
            base_lo = kf['local_bytes']['base']
            n_th = len(kf['thread_per_block']['contribs'])
            n_sh = len(kf['shared_bytes']['contribs'])
            n_lo = len(kf['local_bytes']['contribs'])
            lines.append(f'  K{ki}: th={base_th}(+{n_th}) sh={base_sh}B(+{n_sh}) lo={base_lo}B(+{n_lo})')
        vts_n = len(self.vts_formula['contribs'])
        lines.append(f'  vts: base={self.vts_formula["base"]}(+{vts_n})')
        return '\n'.join(lines)

print('Phase 2: ConstraintSystem loaded.') = ConstraintSystem(recs[0])
    if not cs.spatial_sps:
        cs.built = True; cs.shared_formulas = []
        constraint_systems[wk] = cs
        print(f'T{ti:2d}: skip (no spatial SP)')
        continue
    cs.build()
    constraint_systems[wk] = cs
    ext = [sp['extent'] for sp in cs.spatial_sps]
    s_tiles = [math.prod(sp['lengths']) for sp in cs.spatial_sps]
    r_tiles = [sp['lengths'][0] for sp in cs.reduce_sps]
    est = cs.estimate_shared_memory(s_tiles, r_tiles) if cs.shared_formulas else 0
    print(f'T{ti:2d}: ext={ext} est={est}B')

print(f'\n{len(constraint_systems)} systems built.')

T 0: ext=[4, 4, 49, 256] est=5632B
T 1: ext=[1, 56, 56, 64] est=3840B
T 2: ext=[1, 28, 28, 128] est=14436B
T 3: ext=[1, 1000] est=16016B
T 4: skip (no spatial SP)
T 5: ext=[4, 4, 16, 512] est=4608B
T 6: ext=[1, 7, 7, 512] est=1700B
T 7: ext=[4, 4, 196, 128] est=19968B
T 8: ext=[4, 4, 16, 512] est=2304B
T 9: ext=[4, 4, 196, 128] est=14464B
T10: ext=[1, 112, 112, 64] est=48636B
T11: ext=[1, 28, 28, 128] est=3372B
T12: skip (no spatial SP)
T13: ext=[4, 4, 196, 128] est=13568B
T14: ext=[1, 7, 7, 512] est=540B
T15: skip (no spatial SP)
T16: ext=[4, 4, 49, 256] est=4160B
T17: ext=[6, 6, 196, 64] est=1368B
T18: ext=[1, 14, 14, 256] est=452B
T19: ext=[4, 4, 49, 256] est=5184B
T20: ext=[1, 14, 14, 256] est=1092B
T21: ext=[6, 6, 196, 64] est=10368B
T22: ext=[6, 6, 196, 64] est=6624B
T23: ext=[4, 4, 16, 512] est=16640B

24 systems built.


In [14]:
print('Building constraint systems for all tasks...')
css = {}
for ti, (wk, recs) in enumerate(groups.items()):
    rec = recs[0]
    try:
        cs = ConstraintSystem(rec)
        cs.build()
        css[wk] = cs
        print(f'T{ti:2d}: {cs.summary()}')
    except Exception as e:
        print(f'T{ti:2d}: ERROR {str(e)[:80]}')
print(f'\nTotal: {len(css)}/{len(groups)} tasks built')rmulas:
        print(cs.summary())
        break

Main stage: 6
Spatial SPs: [(4, [2, 2, 1, 1]), (4, [1, 4, 1, 1]), (49, [1, 1, 1, 7]), (256, [1, 4, 1, 1])]
Reduce SPs: [(256, [8, 1])]
Thread bind: 9, PR: 3
Total space: 200704, check_min_thread: True
Shared formulas (2 bufs):
  data_pack.shared: base=1, ['spatial[0] r=2.00', 'spatial[1] r=2.00', 'spatial[2] r=2.00', 'reduce[0] r=2.00']
  p1.shared: base=1, ['spatial[0] r=2.00', 'spatial[1] r=2.00', 'spatial[3] r=2.00', 'reduce[0] r=2.00']


In [15]:
print('=' * 60)
print('Phase 2 검증: formula 정확도 + Phase 3: 랜덤 생성 테스트')
print('=' * 60)

N_SAMPLES = 10
rng = random.Random(42)

total_gen = 0
total_formula_pass = 0
total_verify_pass = 0
total_feat_pass = 0
failed_tasks = []

for ti, (wk, recs) in enumerate(groups.items()):
    if wk not in css:
        continue
    cs = css[wk]
    gen_ok = 0
    verify_ok = 0
    feat_ok = 0

    for si in range(N_SAMPLES):
        rec_gen = cs.generate(rng, max_attempts=500)
        if rec_gen is None:
            continue
        total_gen += 1
        gen_ok += 1

        try:
            task, state = record_to_task_and_state(rec_gen)
        except Exception:
            continue

        # exact lowering check
        exact = verify_state_exact(task, state)
        if exact:
            verify_ok += 1
            total_verify_pass += 1

        # feature extraction check
        try:
            fea = get_per_store_features_from_states([state], task)[0]
            if not (fea.shape[0] == 1 and np.all(fea == 0)):
                feat_ok += 1
                total_feat_pass += 1
        except Exception:
            pass

    rate_v = verify_ok / max(gen_ok, 1) * 100
    rate_f = feat_ok / max(gen_ok, 1) * 100
    status = 'OK' if rate_f >= 90 else 'LOW'
    print(f'T{ti:2d}: gen={gen_ok}/{N_SAMPLES} verify={verify_ok}({rate_v:.0f}%) feat={feat_ok}({rate_f:.0f}%) [{status}]')
    if rate_f < 90:
        failed_tasks.append((ti, wk, gen_ok, verify_ok, feat_ok))

print(f'\n=== 종합 ===')
print(f'Generated: {total_gen}')
print(f'VerifyGPU pass: {total_verify_pass}/{total_gen} ({total_verify_pass/max(total_gen,1)*100:.1f}%)')
print(f'Feature pass:   {total_feat_pass}/{total_gen} ({total_feat_pass/max(total_gen,1)*100:.1f}%)')
if failed_tasks:
    print(f'\nFailed tasks: {len(failed_tasks)}')
    for t in failed_tasks:
        print(f'  T{t[0]}: gen={t[2]} verify={t[3]} feat={t[4]}')
total_valid, total_tested = 0, 0

for ti, (wk, recs) in enumerate(groups.items()):
    cs = constraint_systems[wk]
    if not cs.spatial_sps:
        print(f'T{ti:2d}: skip'); continue

    task_valid = 0
    for trial in range(N_SAMPLES):
        rng = random.Random(42 + ti * 100 + trial)
        try:
            assignments = cs.generate_random_params(rng=rng)
        except RuntimeError:
            total_tested += 1
            print(f'  T{ti:2d} t{trial}: gen failed')
            continue

        new_rec = inject_params(recs[0], assignments)
        ok, detail = validate_via_feature(new_rec)
        total_tested += 1
        if ok:
            task_valid += 1; total_valid += 1
        else:
            sp_a = [(si, v) for si, v in assignments if isinstance(v, list) and len(v) > 1]
            print(f'  T{ti:2d} t{trial}: FAIL {detail}')
            for si, v in sp_a[:4]:
                s = recs[0]['i'][1][1][si]
                if s[0] == 'SP':
                    print(f'    SP[{si}] ext={s[3]} -> {v}')

    print(f'T{ti:2d}: {task_valid}/{N_SAMPLES} valid')

print(f'\n총: {total_valid}/{total_tested} ({total_valid/max(total_tested,1)*100:.1f}%)')

검증: 랜덤 파라미터 (task당 10회)
  T 0 t0: FAIL all-zero
    SP[11] ext=4 -> [1, 1, 4, 1]
    SP[12] ext=4 -> [1, 1, 1, 4]
    SP[13] ext=49 -> [1, 1, 49, 1]
    SP[14] ext=256 -> [1, 16, 1, 1]
  T 0 t2: FAIL all-zero
    SP[11] ext=4 -> [1, 2, 1, 2]
    SP[12] ext=4 -> [1, 1, 1, 4]
    SP[13] ext=49 -> [1, 7, 7, 1]
    SP[14] ext=256 -> [2, 1, 128, 1]
  T 0 t3: FAIL all-zero
    SP[11] ext=4 -> [1, 1, 1, 1]
    SP[12] ext=4 -> [1, 1, 1, 4]
    SP[13] ext=49 -> [1, 7, 7, 1]
    SP[14] ext=256 -> [8, 2, 16, 1]
  T 0 t4: FAIL all-zero
    SP[11] ext=4 -> [4, 1, 1, 1]
    SP[12] ext=4 -> [1, 1, 1, 4]
    SP[13] ext=49 -> [1, 1, 49, 1]
    SP[14] ext=256 -> [1, 4, 1, 2]
  T 0 t5: FAIL all-zero
    SP[11] ext=4 -> [1, 1, 1, 4]
    SP[12] ext=4 -> [1, 2, 1, 1]
    SP[13] ext=49 -> [1, 1, 49, 1]
    SP[14] ext=256 -> [16, 1, 2, 1]
  T 0 t8: FAIL all-zero
    SP[11] ext=4 -> [2, 1, 1, 2]
    SP[12] ext=4 -> [1, 1, 1, 2]
    SP[13] ext=49 -> [1, 1, 49, 1]
    SP[14] ext=256 -> [4, 4, 2, 1]
  T 0 t9: FAI